In [ ]:
from IPython.display import HTML, display

display(HTML("""
<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 60%, #0f3460 100%);
            border-radius: 12px; padding: 40px 40px 30px 40px; margin: 10px 0 30px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <div style="color:#a0aec0; font-size:0.8em; letter-spacing:2px; text-transform:uppercase; margin-bottom:12px;">
    LLM Lab — Section 01c
  </div>
  <h1 style="color:white; font-size:2.2em; margin:0 0 10px 0; font-weight:700; line-height:1.2; border:none;">
    Inference Optimization<br>
    <span style="color:#63b3ed;">Making It Faster</span>
  </h1>
  <p style="color:#cbd5e0; font-size:1em; margin:16px 0 24px 0; max-width:600px; line-height:1.6;">
    First make it work, then make it fast. Five techniques that change how
    efficiently inference runs — without changing what the model produces.
  </p>
  <div style="display:flex; gap:16px; flex-wrap:wrap;">
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">⚡ Bandwidth</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🏃 Speculative</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">📦 Prefix Cache</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🧩 NVFP4</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">⏱ ~20 min</span>
  </div>
</div>
"""))

In [ ]:
# Quick import check — fail fast if wrong kernel
try:
    import openai, psutil
except ImportError as e:
    raise ImportError(
        f"{e}. Select the 'Python 3 (Homebrew)' kernel in Jupyter: "
        "Kernel → Change Kernel → Python 3 (Homebrew)"
    ) from None

import sys, time, re
import html as html_mod, markdown, psutil
from pathlib import Path
from IPython.display import HTML, display

sys.path.insert(0, str(Path('../../scripts').resolve()))
import notebook_helpers

# Discover servers (queries actual model IDs from each port)
MODELS, clients = notebook_helpers.discover_servers()

if not MODELS:
    raise RuntimeError("No MLX servers found! Start at least one on ports 8800-8809.")

# Warm up with live-updating status table
notebook_helpers.warmup_models(MODELS, clients)
notebook_helpers.init(MODELS, clients)

In [ ]:
# Import shared helper functions
from notebook_helpers import strip_think, _md, _render_cards, compare_models, show_metrics_table
print("Helpers loaded.")


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">⚡ Step 1: Memory Bandwidth Is the Bottleneck</h2>
</div>

LLM decode is **memory-bandwidth-bound**, not compute-bound. Each token requires reading the model's active weights from RAM. The M4 Max has 546 GB/s of memory bandwidth — that's the speed limit.

**The embedded analogy:** This is like reading flash at bus speed. You can't go faster than the bus, no matter how fast your CPU is. The ALU sits idle waiting for data.

In [ ]:
# Calculate theoretical tok/s from M4 Max bandwidth
# Then compare against measured tok/s for each model

BW_GBS = 546  # M4 Max memory bandwidth in GB/s
BYTES_PER_PARAM = 0.5  # 4-bit quantization

# Build specs dynamically from discovered MODELS
model_specs = []
for m in MODELS:
    active_b = notebook_helpers.active_params_b(m["label"])
    active_gb = active_b * BYTES_PER_PARAM
    is_moe = "-A" in m["label"]
    # Parse total params from label for MoE description
    total_match = re.match(r'^(\d+(?:\.\d+)?)B', m["label"])
    total_b = float(total_match.group(1)) if total_match else active_b
    if is_moe:
        desc = f"MoE: ~{active_b:.0f}B active × {BYTES_PER_PARAM} B/param"
    else:
        desc = f"Dense: {total_b:.0f}B × {BYTES_PER_PARAM} B/param"
    model_specs.append({"label": m["label"], "color": m["color"], "active_gb": active_gb, "desc": desc})

# Measure actual tok/s with a standard prompt
print("Measuring actual throughput...", flush=True)
measured = {}

def measure_tps(m):
    t0 = time.time()
    r = clients[m["label"]].chat.completions.create(
        model=m["model"],
        messages=[{"role": "user", "content": "Write a 100-word paragraph about resistor color codes."}],
        max_tokens=200,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    elapsed = time.time() - t0
    tokens = r.usage.completion_tokens
    return m["label"], tokens / elapsed if elapsed > 0 else 0

with concurrent.futures.ThreadPoolExecutor(max_workers=len(MODELS)) as ex:
    for label, tps in ex.map(measure_tps, MODELS):
        measured[label] = tps

# Build comparison table
rows = ""
for spec in model_specs:
    theoretical = BW_GBS / spec["active_gb"] if spec["active_gb"] > 0 else 0
    actual = measured.get(spec["label"], 0)
    efficiency = (actual / theoretical * 100) if theoretical > 0 else 0
    
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:{spec['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{spec['label']}</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>{spec['active_gb']:.1f} GB</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>{theoretical:.0f} tok/s</td>"
        f"<td style='padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{actual:.1f} tok/s</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>{efficiency:.0f}%</td>"
        f"<td style='padding:6px 12px; color:#6b7280; font-size:0.85em; border-bottom:1px solid #e5e7eb;'>{spec['desc']}</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:700px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Active Wt</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Theoretical</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Measured</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Efficiency</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Notes</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
<div style="color:#6b7280; font-size:0.8em; margin-top:4px;">
  Theoretical max = {BW_GBS} GB/s ÷ active weight bytes per token.
  Gap between theoretical and measured = attention compute + KV cache reads + scheduling overhead.
</div>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🏃 Step 2: Speculative Decoding</h2>
</div>

Autoregressive decoding is inherently sequential: each token depends on the previous one. Speculative decoding breaks this bottleneck.

**The trick:** A small, fast "draft" model guesses several tokens ahead. The big "target" model verifies all guesses in a single forward pass. Accepted tokens are kept; rejected ones get regenerated. The output is *mathematically identical* to standard decoding.

**The embedded analogy:** This is **branch prediction** in a CPU pipeline. The processor speculatively executes down the predicted branch. If right, you saved cycles. If wrong, you flush and redo — but the hit rate makes it a net win.

In [ ]:
# Speculative decoding math — show the theoretical speedup
# Use measured throughput from the bandwidth cell to derive decode times

# Derive decode time per token from measured throughput (ms/tok = 1000/tok_s)
decode_ms = {}
for spec in model_specs:
    tps = measured.get(spec["label"], 0)
    decode_ms[spec["label"]] = round(1000 / tps, 1) if tps > 0 else 999

# Draft model: assume a ~1.5B model at ~3ms/token
draft_ms = 3
num_draft = 4  # guess 4 tokens ahead

# Find slowest and fastest models for the explanation
slowest = max(MODELS, key=lambda m: decode_ms.get(m["label"], 0))
fastest = min(MODELS, key=lambda m: decode_ms.get(m["label"], 999))
slow_ms = decode_ms[slowest["label"]]

display(HTML(f"""
<div style="background:#f9fafb; border:1px solid #d1d5db; border-radius:8px; padding:16px 20px; margin:10px 0; font-family:monospace;">
  <div style="font-weight:bold; color:#1e3a5f; margin-bottom:12px;">Standard Decode (sequential)</div>
  <pre style="margin:0; color:#374151;">
Token 1 → [forward pass] → Token 2 → [forward pass] → Token 3 → ...
Each step reads the full active weights from memory.
  </pre>
  <div style="font-weight:bold; color:#1e3a5f; margin:16px 0 12px 0;">Speculative Decode (draft + verify)</div>
  <pre style="margin:0; color:#374151;">
Draft:  Token 1,2,3,4 → [4 cheap passes]  → ~{num_draft * draft_ms}ms total
Target: verify all 4  → [1 forward pass]   → ~{slow_ms:.0f}ms ({slowest['label']})
Accept 3 of 4 → regenerate 1
4 tokens in ~{num_draft * draft_ms + slow_ms:.0f}ms vs ~{num_draft * slow_ms:.0f}ms standard = {num_draft * slow_ms / (num_draft * draft_ms + slow_ms):.1f}× speedup
  </pre>
</div>
"""))

# Calculate speedup for each model
rows = ""
for m in MODELS:
    target_ms = decode_ms[m["label"]]
    # Standard: N tokens = N * target_ms
    standard_time = num_draft * target_ms
    # Speculative: draft time + 1 verify pass + ~1 regen
    # Assume 75% acceptance rate
    accepted = int(num_draft * 0.75)
    spec_time = (num_draft * draft_ms) + target_ms + (num_draft - accepted) * target_ms
    speedup = standard_time / spec_time if spec_time > 0 else 0
    benefit = "High" if speedup > 2 else "Moderate" if speedup > 1.3 else "Low"
    benefit_color = "#16a34a" if speedup > 2 else "#f59e0b" if speedup > 1.3 else "#dc2626"
    
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:{m['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{m['label']}</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>{target_ms:.0f}ms</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>{standard_time:.0f}ms</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>{spec_time:.0f}ms</td>"
        f"<td style='padding:6px 12px; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{speedup:.1f}×</td>"
        f"<td style='padding:6px 12px; color:{benefit_color}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{benefit}</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:600px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Target</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Decode/tok</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Standard ({num_draft} tok)</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Speculative</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Speedup</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Benefit</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
<div style="color:#6b7280; font-size:0.8em; margin-top:4px;">
  Assumes ~1.5B draft model at {draft_ms}ms/tok, 75% acceptance rate, {num_draft} draft tokens.
  Decode/tok derived from measured throughput. Bigger benefit when target model is slow.
</div>
"""))

print(f"\nKey insight: speculation helps most with slow target models ({slowest['label']} at {slow_ms:.0f}ms/tok).")
print(f"For the fast {fastest['label']} model, the draft overhead may not be worth it.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📦 Step 3: Prefix Caching</h2>
</div>

When multiple requests share the same system prompt, the server recomputes the same KV cache entries every time. **Prefix caching** reuses the KV state from previously computed prefixes.

This extends the KV cache demo from Section 01 with a practical application: measure TTFT cold vs cached across all 3 models.

In [ ]:
# Prefix caching: send same system prompt twice, measure TTFT improvement

# Build a substantial system prompt (~500 tokens)
SYSTEM_PROMPT = (
    "You are an expert embedded systems engineer with deep knowledge of ARM Cortex-M "
    "microcontrollers, RTOS design, peripheral drivers, and hardware debugging. "
    "You explain concepts clearly using hardware analogies. "
    "Always mention relevant register names and memory addresses when discussing peripherals. "
) * 10  # Repeat to make the prefix substantial

def measure_ttft(client, model_name, system_prompt, user_msg):
    """Measure time-to-first-token with streaming."""
    t0 = time.time()
    stream = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg},
        ],
        max_tokens=1,
        stream=True,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            return time.time() - t0
    return time.time() - t0

print("Measuring prefix caching TTFT (cold vs cached)...", flush=True)
print(f"System prompt length: ~{len(SYSTEM_PROMPT.split())} words\n")

results = []
for m in MODELS:
    print(f"  Testing {m['label']}...", flush=True)
    # Cold: first request with this system prompt
    cold = measure_ttft(clients[m["label"]], m["model"], SYSTEM_PROMPT, "What is DMA?")
    # Cached: same system prompt, different question
    cached = measure_ttft(clients[m["label"]], m["model"], SYSTEM_PROMPT, "What is SPI?")
    speedup = cold / cached if cached > 0 else 0
    results.append({"label": m["label"], "color": m["color"], "cold": cold, "cached": cached, "speedup": speedup})

rows = ""
for r in results:
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:{r['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['label']}</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>{r['cold']*1000:.0f} ms</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>{r['cached']*1000:.0f} ms</td>"
        f"<td style='padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{r['speedup']:.1f}\u00d7</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">TTFT Cold</th>
    <th style="padding:8px 12px; color:white; text-align:left;">TTFT Cached</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Speedup</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
<div style="color:#6b7280; font-size:0.8em; margin-top:4px;">
  Same system prompt, different user question. The server reuses KV cache for the shared prefix.
  Speedup scales with prefix length \u2014 longer shared prompts = bigger savings.
</div>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧩 Step 4: Quantization Format Deep Dive</h2>
</div>

Section 01 covered quantization basics. Here we go deeper: **not all 4-bit is equal.**

MLX supports three 4-bit modes:
- **Affine (INT4)**: Uniform integer grid. Like a uniform-resolution ADC.
- **MXFP4**: E2M1 floating-point with E8M0 power-of-2 scale. Like a log-scale ADC.
- **NVFP4**: E2M1 floating-point with E4M3 float scale, group size 16. Like a floating-point ADC with per-channel calibration.

**The embedded analogy:** NVFP4 is a floating-point ADC — non-uniform quantization levels that cluster where the signal lives (near zero for neural network weights). INT4 is a uniform ADC — equal spacing wastes resolution at the extremes.

In [ ]:
# Visualize the quantization level distributions

# E2M1 representable values (what NVFP4 and MXFP4 use)
e2m1_values = [0, 0.5, 1, 1.5, 2, 3, 4, 6, -0.5, -1, -1.5, -2, -3, -4, -6]
e2m1_positive = sorted([v for v in e2m1_values if v >= 0])

# INT4 affine: 16 uniform levels across a typical range [-6, 6]
int4_levels = [round(-6 + i * 12/15, 2) for i in range(16)]
int4_positive = sorted([v for v in int4_levels if v >= 0])

# Build visual comparison
def level_viz(values, label, color, max_val=6.5):
    """Create a number line visualization of quantization levels."""
    dots = ""
    for v in sorted(values):
        pct = (v + max_val) / (2 * max_val) * 100
        dots += f'<div style="position:absolute; left:{pct}%; top:50%; transform:translate(-50%,-50%); width:8px; height:8px; background:{color}; border-radius:50%;" title="{v}"></div>'
    return f"""
    <div style="margin:8px 0;">
      <div style="color:{color}; font-weight:bold; font-size:0.85em; margin-bottom:4px;">{label}</div>
      <div style="position:relative; height:24px; background:#f3f4f6; border:1px solid #d1d5db; border-radius:4px;">
        <div style="position:absolute; left:50%; top:0; bottom:0; width:1px; background:#9ca3af;"></div>
        {dots}
      </div>
    </div>"""

display(HTML(f"""
<div style="background:#f9fafb; border:1px solid #d1d5db; border-radius:8px; padding:16px 20px; margin:10px 0;">
  <div style="font-weight:bold; color:#1e3a5f; margin-bottom:8px;">Quantization Level Distribution</div>
  <div style="font-size:0.8em; color:#6b7280; margin-bottom:12px;">Each dot = one representable value. Center = 0. Neural network weights cluster near zero.</div>
  {level_viz(e2m1_values, 'NVFP4 / MXFP4 (E2M1) \u2014 denser near zero', '#16a34a')}
  {level_viz(int4_levels, 'Affine INT4 \u2014 uniform spacing', '#dc2626')}
</div>
"""))

# Format comparison table
display(HTML("""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:500px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Format</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Weight Fmt</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Scale Type</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Group Size</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Quality</th>
  </tr></thead>
  <tbody>
    <tr><td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">Affine</td>
        <td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">INT4</td>
        <td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">fp16 (per group)</td>
        <td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">64</td>
        <td style="padding:6px 12px; color:#dc2626; font-weight:bold; border-bottom:1px solid #e5e7eb;">Lowest</td></tr>
    <tr><td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">MXFP4</td>
        <td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">E2M1</td>
        <td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">E8M0 (power-of-2)</td>
        <td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">32</td>
        <td style="padding:6px 12px; color:#f59e0b; font-weight:bold; border-bottom:1px solid #e5e7eb;">Middle</td></tr>
    <tr><td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">NVFP4</td>
        <td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">E2M1</td>
        <td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">E4M3 float</td>
        <td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">16</td>
        <td style="padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;">Highest</td></tr>
  </tbody>
</table>
"""))

print("Group size = how many weights share one scale factor.")
print("Like calibrating an ADC: shared cal across 64 channels vs per-16-channel cal.")
print("Smaller groups = more independent calibrations = less systematic error.")

In [ ]:
# Compare response quality across all 3 models on a precision-sensitive prompt
# All 3 servers run NVFP4 — ask them about quantization itself

results = compare_models(
    "Explain the difference between NVFP4 and affine INT4 quantization in exactly 3 sentences. "
    "Use an ADC analogy.",
    max_tokens=512,
)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🔄 Step 5: Continuous Batching</h2>
</div>

`mlx_lm.server` processes requests **serially** — one at a time. Send 3 concurrent requests and the 2nd waits for the 1st to finish. This is fine for personal use but matters for multi-user serving.

Let's make this observable.

In [ ]:
# Send concurrent requests to the SAME server and observe serial behavior
# Pick the slowest model (most interesting for demonstrating serial queueing)

target = max(MODELS, key=lambda m: decode_ms.get(m["label"], 0))
client = clients[target["label"]]

prompts = [
    "Count from 1 to 20, one number per line.",
    "Count from 21 to 40, one number per line.",
    "Count from 41 to 60, one number per line.",
]

def timed_request(idx, prompt):
    t0 = time.time()
    r = client.chat.completions.create(
        model=target["model"],
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    elapsed = time.time() - t0
    return idx, elapsed, r.usage.completion_tokens

print(f"Sending 3 concurrent requests to {target['label']} (port {target['port']})...\n", flush=True)
t_start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=3) as ex:
    futures = [ex.submit(timed_request, i, p) for i, p in enumerate(prompts)]
    req_results = [f.result() for f in concurrent.futures.as_completed(futures)]

total_wall = time.time() - t_start
req_results.sort(key=lambda x: x[0])

# Build timeline visualization
bars = ""
colors = ["#2563eb", "#16a34a", "#f59e0b"]
sum_individual = sum(r[1] for r in req_results)

for idx, elapsed, tokens in req_results:
    width_pct = (elapsed / total_wall) * 100
    bars += f"""
    <div style="display:flex; align-items:center; margin:4px 0;">
      <div style="width:80px; font-size:0.8em; color:#6b7280;">Request {idx+1}</div>
      <div style="flex:1; background:#f3f4f6; border-radius:4px; height:24px; position:relative;">
        <div style="background:{colors[idx % len(colors)]}; height:100%; width:{width_pct}%; border-radius:4px; opacity:0.8;"></div>
        <span style="position:absolute; right:8px; top:50%; transform:translateY(-50%); font-size:0.75em; color:#374151;">{elapsed:.1f}s / {tokens} tok</span>
      </div>
    </div>"""

display(HTML(f"""
<div style="background:#f9fafb; border:1px solid #d1d5db; border-radius:8px; padding:16px 20px; margin:10px 0;">
  <div style="font-weight:bold; color:#1e3a5f; margin-bottom:8px;">Concurrent Request Timeline ({target['label']})</div>
  {bars}
  <div style="margin-top:12px; font-size:0.8em; color:#6b7280;">
    Wall clock: {total_wall:.1f}s &nbsp;|&nbsp; Sum of individual: {sum_individual:.1f}s &nbsp;|&nbsp;
    Overlap: {(1 - total_wall / sum_individual) * 100:.0f}%
  </div>
  <div style="margin-top:4px; font-size:0.8em; color:#6b7280;">
    If batched: wall clock ≈ longest single request. If serial: wall clock ≈ sum of all.
  </div>
</div>
"""))

if total_wall > sum_individual * 0.7:
    print("→ Requests processed mostly serially (expected for mlx_lm.server).")
    print("  For personal use this is fine. For multi-user serving, you'd need continuous batching.")
else:
    print("→ Some parallelism detected! Server may support basic batching.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📝 Takeaways</h2>
</div>

**Optimization checklist (biggest returns first):**

1. **Memory bandwidth is the wall** — decode throughput = bandwidth \u00f7 active weight bytes. You can't go faster than the bus
2. **Speculative decoding** trades cheap draft passes for expensive target passes. 2-4\u00d7 speedup on slow models (122B). Like branch prediction \u2014 biggest wins on deep pipelines
3. **Prefix caching** eliminates redundant prefill. If your requests share a system prompt, you only pay once. TTFT improvement scales with prefix length
4. **Not all 4-bit is equal** \u2014 NVFP4 > MXFP4 > Affine at the same bit width. Floating-point formats match weight distributions better. Smaller group sizes = less quantization error
5. **Continuous batching** matters for multi-user serving, not personal use. `mlx_lm.server` is serial \u2014 fine for one user

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 What's Next</h2>
</div>

- **Section 02:** Structured output — making LLMs return JSON you can actually parse
- **Section 03:** Tool use — letting models call functions and interact with external systems